# EKF Analysis Notebook

This notebook provides an interactive interface for analyzing robot localization data
using the Python EKF implementation.

## Contents
1. Setup and Imports
2. Load Your Data
3. Configure the EKF
4. Run the Filter
5. Visualize Results
6. Export Filtered States

## 1. Setup and Imports

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional
import json

# Import EKF modules
from data_types import (
    Imu, Odometry, EKFState, Measurement,
    STATE_SIZE, STATE_X, STATE_Y, STATE_Z,
    STATE_ROLL, STATE_PITCH, STATE_YAW,
    STATE_VX, STATE_VY, STATE_VZ
)
from ekf import EKF, EKFConfig

# Plot settings
%matplotlib inline
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print("Imports complete!")

## 2. Load Your Data

Modify this section to load your own sensor data. The data should be converted
to the `Imu` and `Odometry` dataclasses.

In [ ]:
# Example: Load from CSV or your custom format
# Uncomment and modify as needed

def load_imu_from_csv(filepath: str) -> List[Imu]:
    """
    Load IMU data from CSV file.
    
    Expected columns: timestamp, gx, gy, gz, ax, ay, az, qx, qy, qz, qw
    """
    # data = np.genfromtxt(filepath, delimiter=',', skip_header=1)
    # imu_list = []
    # for row in data:
    #     imu = Imu(
    #         timestamp=row[0],
    #         angular_velocity=row[1:4],
    #         linear_acceleration=row[4:7],
    #         orientation=Rotation.from_quat(row[7:11]),
    #         angular_velocity_covariance=np.eye(3) * 0.01**2,
    #         linear_acceleration_covariance=np.eye(3) * 0.1**2,
    #         orientation_covariance=np.eye(3) * 0.02**2
    #     )
    #     imu_list.append(imu)
    # return imu_list
    pass

def load_odom_from_csv(filepath: str) -> List[Odometry]:
    """
    Load odometry data from CSV file.
    
    Expected columns: timestamp, x, y, z, qx, qy, qz, qw, vx, vy, vz, wx, wy, wz
    """
    # data = np.genfromtxt(filepath, delimiter=',', skip_header=1)
    # odom_list = []
    # for row in data:
    #     odom = Odometry(
    #         timestamp=row[0],
    #         position=row[1:4],
    #         orientation=Rotation.from_quat(row[4:8]),
    #         linear_velocity=row[8:11],
    #         angular_velocity=row[11:14],
    #         pose_covariance=np.eye(6) * 0.05**2,
    #         twist_covariance=np.eye(6) * 0.1**2
    #     )
    #     odom_list.append(odom)
    # return odom_list
    pass

In [ ]:
# For demonstration, generate synthetic data
# Replace this with your actual data loading

from example import (
    generate_circular_trajectory,
    generate_imu_measurements,
    generate_odometry_measurements
)

np.random.seed(42)

# Generate synthetic data
timestamps, ground_truth = generate_circular_trajectory(
    duration=20.0,
    dt=0.01,
    radius=5.0,
    angular_velocity=0.5
)

imu_measurements = generate_imu_measurements(
    ground_truth,
    rate=100.0,
    gyro_noise_std=0.01,
    accel_noise_std=0.1,
    orientation_noise_std=0.02
)

odom_measurements = generate_odometry_measurements(
    ground_truth,
    rate=20.0,
    position_noise_std=0.05,
    orientation_noise_std=0.02,
    velocity_noise_std=0.1
)

print(f"Loaded {len(imu_measurements)} IMU measurements")
print(f"Loaded {len(odom_measurements)} odometry measurements")
print(f"Time range: {imu_measurements[0].timestamp:.2f}s to {imu_measurements[-1].timestamp:.2f}s")

## 3. Configure the EKF

Adjust the process noise covariance and other parameters to tune the filter.

In [ ]:
# Process noise covariance (Q matrix)
# Higher values = filter adapts faster but is noisier
# Lower values = filter is smoother but responds slower

process_noise = np.diag([
    0.05,   # x position
    0.05,   # y position
    0.06,   # z position
    0.03,   # roll
    0.03,   # pitch
    0.06,   # yaw
    0.025,  # x velocity
    0.025,  # y velocity
    0.04,   # z velocity
    0.01,   # roll velocity
    0.01,   # pitch velocity
    0.02,   # yaw velocity
    0.01,   # x acceleration
    0.01,   # y acceleration
    0.015   # z acceleration
])

# Initial estimate covariance
initial_covariance = np.eye(STATE_SIZE) * 0.1

# Create configuration
config = EKFConfig(
    process_noise_covariance=process_noise,
    initial_covariance=initial_covariance,
    two_d_mode=False,
    use_dynamic_process_noise=False,
    mahalanobis_threshold=5.0
)

print("EKF configuration created")
print(f"  Process noise diagonal: {np.diag(process_noise)[:6]}... (showing first 6)")
print(f"  Mahalanobis threshold: {config.mahalanobis_threshold}")

## 4. Run the Filter

In [ ]:
# Initialize EKF
ekf = EKF(config)

# Set initial state (use first measurement or known starting point)
initial_state = ground_truth[0].copy()  # Replace with your initial state
initial_state.covariance = initial_covariance
ekf.set_state(initial_state)

# Enable history recording
ekf.enable_history(True)

print(f"EKF initialized at t={initial_state.timestamp:.3f}s")
print(f"Initial position: {initial_state.position}")

In [ ]:
# Combine and sort measurements by timestamp
all_measurements = []
for imu in imu_measurements:
    all_measurements.append(('imu', imu))
for odom in odom_measurements:
    all_measurements.append(('odom', odom))

all_measurements.sort(key=lambda x: x[1].timestamp)

print(f"Processing {len(all_measurements)} total measurements...")

# Process measurements
accepted_count = 0
rejected_count = 0

for meas_type, meas in all_measurements:
    if meas_type == 'imu':
        accepted = ekf.correct_imu(
            meas,
            update_orientation=True,
            update_angular_velocity=True,
            update_linear_acceleration=True,
            remove_gravity=True
        )
        if accepted:
            accepted_count += 1
        else:
            rejected_count += 1
            
    elif meas_type == 'odom':
        # Configure which states to update from odometry
        pose_update = np.array([True, True, False, False, False, True])  # x, y, yaw
        twist_update = np.array([True, True, False, False, False, True])  # vx, vy, vyaw
        
        pose_accepted, twist_accepted = ekf.correct_odometry(
            meas, 
            pose_update, 
            twist_update
        )
        if pose_accepted and twist_accepted:
            accepted_count += 1
        else:
            rejected_count += 1

# Get filtered states
filtered_states = ekf.get_history()

print(f"\nProcessing complete!")
print(f"  Accepted measurements: {accepted_count}")
print(f"  Rejected measurements: {rejected_count}")
print(f"  Output states: {len(filtered_states)}")

## 5. Visualize Results

In [ ]:
# Extract data for plotting
gt_times = np.array([s.timestamp for s in ground_truth])
gt_x = np.array([s.position[0] for s in ground_truth])
gt_y = np.array([s.position[1] for s in ground_truth])
gt_yaw = np.array([s.yaw for s in ground_truth])

filt_times = np.array([s.timestamp for s in filtered_states])
filt_x = np.array([s.position[0] for s in filtered_states])
filt_y = np.array([s.position[1] for s in filtered_states])
filt_yaw = np.array([s.yaw for s in filtered_states])

odom_times = np.array([o.timestamp for o in odom_measurements])
odom_x = np.array([o.position[0] for o in odom_measurements])
odom_y = np.array([o.position[1] for o in odom_measurements])

In [ ]:
# Plot XY trajectory
fig, ax = plt.subplots(figsize=(10, 10))

ax.plot(gt_x, gt_y, 'g-', label='Ground Truth', linewidth=2)
ax.plot(filt_x, filt_y, 'b--', label='Filtered', linewidth=1.5)
ax.scatter(odom_x, odom_y, c='r', s=10, alpha=0.5, label='Odometry')

ax.set_xlabel('X (m)', fontsize=12)
ax.set_ylabel('Y (m)', fontsize=12)
ax.set_title('XY Trajectory Comparison', fontsize=14)
ax.legend(fontsize=10)
ax.axis('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot position over time
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# X position
axes[0].plot(gt_times, gt_x, 'g-', label='Ground Truth', linewidth=2)
axes[0].plot(filt_times, filt_x, 'b--', label='Filtered', linewidth=1.5)
axes[0].scatter(odom_times, odom_x, c='r', s=5, alpha=0.5, label='Odometry')
axes[0].set_ylabel('X (m)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Y position
axes[1].plot(gt_times, gt_y, 'g-', label='Ground Truth', linewidth=2)
axes[1].plot(filt_times, filt_y, 'b--', label='Filtered', linewidth=1.5)
axes[1].scatter(odom_times, odom_y, c='r', s=5, alpha=0.5, label='Odometry')
axes[1].set_ylabel('Y (m)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Yaw
axes[2].plot(gt_times, np.rad2deg(gt_yaw), 'g-', label='Ground Truth', linewidth=2)
axes[2].plot(filt_times, np.rad2deg(filt_yaw), 'b--', label='Filtered', linewidth=1.5)
axes[2].set_ylabel('Yaw (deg)')
axes[2].set_xlabel('Time (s)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Position and Orientation Over Time', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compute and plot errors
from example import compute_errors

pos_err, vel_err, ori_err = compute_errors(ground_truth, filtered_states)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

# Position error
axes[0].plot(filt_times, pos_err, 'b-', linewidth=1)
axes[0].axhline(np.mean(pos_err), color='r', linestyle='--', label=f'Mean: {np.mean(pos_err):.3f} m')
axes[0].set_ylabel('Position Error (m)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Velocity error
axes[1].plot(filt_times, vel_err, 'b-', linewidth=1)
axes[1].axhline(np.mean(vel_err), color='r', linestyle='--', label=f'Mean: {np.mean(vel_err):.3f} m/s')
axes[1].set_ylabel('Velocity Error (m/s)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Orientation error
axes[2].plot(filt_times, np.rad2deg(ori_err), 'b-', linewidth=1)
axes[2].axhline(np.rad2deg(np.mean(ori_err)), color='r', linestyle='--', 
                label=f'Mean: {np.rad2deg(np.mean(ori_err)):.3f} deg')
axes[2].set_ylabel('Orientation Error (deg)')
axes[2].set_xlabel('Time (s)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Filter Errors Over Time', fontsize=14)
plt.tight_layout()
plt.show()

# Print error statistics
print("\nError Statistics:")
print(f"  Position: {np.mean(pos_err):.4f} +/- {np.std(pos_err):.4f} m (max: {np.max(pos_err):.4f})")
print(f"  Velocity: {np.mean(vel_err):.4f} +/- {np.std(vel_err):.4f} m/s (max: {np.max(vel_err):.4f})")
print(f"  Orientation: {np.rad2deg(np.mean(ori_err)):.4f} +/- {np.rad2deg(np.std(ori_err)):.4f} deg")

## 6. Export Filtered States

In [ ]:
def export_to_csv(states: List[EKFState], filepath: str) -> None:
    """
    Export filtered states to CSV file.
    """
    header = "timestamp,x,y,z,roll,pitch,yaw,vx,vy,vz,wx,wy,wz,ax,ay,az"
    
    with open(filepath, 'w') as f:
        f.write(header + '\n')
        for state in states:
            rpy = state.orientation.as_euler('xyz')
            line = ','.join([
                f"{state.timestamp:.6f}",
                f"{state.position[0]:.6f}",
                f"{state.position[1]:.6f}",
                f"{state.position[2]:.6f}",
                f"{rpy[0]:.6f}",
                f"{rpy[1]:.6f}",
                f"{rpy[2]:.6f}",
                f"{state.linear_velocity[0]:.6f}",
                f"{state.linear_velocity[1]:.6f}",
                f"{state.linear_velocity[2]:.6f}",
                f"{state.angular_velocity[0]:.6f}",
                f"{state.angular_velocity[1]:.6f}",
                f"{state.angular_velocity[2]:.6f}",
                f"{state.linear_acceleration[0]:.6f}",
                f"{state.linear_acceleration[1]:.6f}",
                f"{state.linear_acceleration[2]:.6f}"
            ])
            f.write(line + '\n')
    
    print(f"Exported {len(states)} states to {filepath}")

# Export the filtered states
# export_to_csv(filtered_states, 'filtered_output.csv')

In [ ]:
def export_to_json(states: List[EKFState], filepath: str) -> None:
    """
    Export filtered states to JSON file (includes covariance).
    """
    data = []
    for state in states:
        rpy = state.orientation.as_euler('xyz')
        quat = state.orientation.as_quat()
        data.append({
            'timestamp': state.timestamp,
            'position': state.position.tolist(),
            'orientation_euler': rpy.tolist(),
            'orientation_quat': quat.tolist(),
            'linear_velocity': state.linear_velocity.tolist(),
            'angular_velocity': state.angular_velocity.tolist(),
            'linear_acceleration': state.linear_acceleration.tolist(),
            'covariance_diagonal': np.diag(state.covariance).tolist()
        })
    
    with open(filepath, 'w') as f:
        json.dump(data, f, indent=2)
    
    print(f"Exported {len(states)} states to {filepath}")

# Export to JSON
# export_to_json(filtered_states, 'filtered_output.json')

## 7. Inspect Individual States (Optional)

In [ ]:
# Inspect a specific state
state_idx = len(filtered_states) // 2  # Middle state
state = filtered_states[state_idx]

print(f"State at index {state_idx}:")
print(f"  Timestamp: {state.timestamp:.3f} s")
print(f"  Position: [{state.position[0]:.3f}, {state.position[1]:.3f}, {state.position[2]:.3f}] m")
print(f"  Orientation (RPY): [{np.rad2deg(state.roll):.2f}, {np.rad2deg(state.pitch):.2f}, {np.rad2deg(state.yaw):.2f}] deg")
print(f"  Linear velocity: [{state.linear_velocity[0]:.3f}, {state.linear_velocity[1]:.3f}, {state.linear_velocity[2]:.3f}] m/s")
print(f"  Angular velocity: [{state.angular_velocity[0]:.3f}, {state.angular_velocity[1]:.3f}, {state.angular_velocity[2]:.3f}] rad/s")
print(f"  Covariance diagonal (position): {np.diag(state.covariance)[:3]}")

In [ ]:
# Plot covariance evolution
cov_x = [np.sqrt(s.covariance[STATE_X, STATE_X]) for s in filtered_states]
cov_y = [np.sqrt(s.covariance[STATE_Y, STATE_Y]) for s in filtered_states]
cov_yaw = [np.sqrt(s.covariance[STATE_YAW, STATE_YAW]) for s in filtered_states]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(filt_times, cov_x, 'b-')
axes[0].set_ylabel('X Std Dev (m)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(filt_times, cov_y, 'b-')
axes[1].set_ylabel('Y Std Dev (m)')
axes[1].grid(True, alpha=0.3)

axes[2].plot(filt_times, np.rad2deg(cov_yaw), 'b-')
axes[2].set_ylabel('Yaw Std Dev (deg)')
axes[2].set_xlabel('Time (s)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Filter Covariance (Standard Deviation) Over Time', fontsize=14)
plt.tight_layout()
plt.show()